| 步骤 | 数学表达式 | 解释 |
|------|-----------|------|
| 1. 线性假设 | $( y = w^T x + b )$ | 特征线性组合 |
| 2. 概率映射 | $( \hat{y} = \rm{sigmoid}(y) = \dfrac{1}{1+e^{-z}} )$ | 压缩到 `[0,1]`，成为概率 |
| 3. 标签分布 | $( y \mid x \sim \text{Bernoulli}(\hat{y}) )$ | 伯努利分布假设 |
| 4. 目标函数 | 最大化似然 | 等价于最小化交叉熵 |
| 5. 优化求解 | 梯度下降 | 梯度形式：$( \sum (\hat{y}_i - y_i)x_i )$ |

# 逻辑回归的实现

## 1.模块引入

In [1]:
import torch # 引入张量
from sklearn.datasets import load_iris # 数据集
# torch.manual_seed(88) # 随机种子，不设置会使用当前时间作为参数，每次都不同
# 线性回归的随机参数是靠随机种子通过随机算法生成的，如果不设置随机种子，那每次拿到的数据参数获得的结果每次都不同

In [2]:
# s_data = load_iris()

In [3]:
# s_data['target'] # 0,1完全线性可分， 2与0，1不可分

In [5]:
data, target = load_iris(return_X_y = True)

In [6]:
x = torch.Tensor(data[0: 100])
y = torch.Tensor(target[0: 100]).view(100, 1)

## 2. 准备数据集

## 3. 定义权重

In [7]:
w = torch.randn(1, 4) # 神经元个数1
b = torch.randn(1) # 偏置值

w.requires_grad = True
b.requires_grad = True # 指定回归函数参数可被求导 -> 可训练

## 4. 矩阵运算(线性回归)

In [8]:
def linear(x: torch.Tensor, w:torch.Tensor, b:torch.Tensor) -> torch.Tensor:
    return x @ w.T + b

In [16]:
# x = torch.Tensor([1.0, 0.2, 2.0, 3.1])
# linear(x, w, b)

## 5. sigmoid激活函数的实现(softmax)

In [9]:
def sigmoid(y: torch.Tensor) -> torch.Tensor:
    return 1 / (1 + torch.exp(-y))

In [18]:
x = torch.Tensor([1.0, 0.2, 2.0, 3.1])
y = linear(x, w, b)

sigmoid(y)

tensor([0.0648], grad_fn=<MulBackward0>)

## 6. 极大似然导出的损失函数(binary-cross-entroy)

In [10]:
def binary_cross_entropy(y_pred: torch.Tensor, y_true: torch.Tensor) -> torch.Tensor:
    _loss = -(y_true * torch.log(y_pred) + (1 - y_true) * torch.log(1 - y_pred))
    return _loss.mean()

## 7. 梯度下降公式实现

In [ ]:
# 循环执行，不断更新w，b，直到将样本分开。
epoch = 10000 # 训练次数
learning_rate = 0.001 # 学习率， 通常在10的-3到-5次方
for e in range(epoch):
    # 1. 线性运算，预测
    y_ = linear(x, w, b)
    y_ = sigmoid(y_)
    # 2. 计算误差
    loss = binary_cross_entropy(y_, y)
    # 3. 自动求导
    loss.backward()
    # 4. 梯度下降公式反向传导调整参数
    with torch.no_grad(): # ??
        w -= learning_rate * w.grad
        b -= learning_rate * b.grad
        # ??
        w.grad.zero_()
        b.grad.zero_()
    # 5. 测试更新后的神经元，测准确率
    if e % 100 == 0:
        print(f"loss损失值：{loss:10.6f}", end="", flush=True)
        y_[ y_ > 0.5 ] = 1
        y_[ y_ <= 0.5 ] = 0
        current_rate = (y_ == y).float().mean()
        print(f"准确率:{current_rate * 100:.2f}")
print("训练完毕")


In [7]:
# 终端美化，进度条样式
from tqdm import tqdm
import time

ls = [1, 2, 3, 4, 5, 6, 7]
tm = tqdm(ls, ncols=100, total=7, colour="blue", unit="bit")
for n in tm:
    time.sleep(0.5)

100%|████████████████████████████████████████████████████████████████| 7/7 [00:03<00:00,  1.99bit/s]


In [ ]:
import torch # 引入张量
from sklearn.datasets import load_iris # 数据集
# torch.manual_seed(88) # 随机种子，不设置会使用当前时间作为参数，每次都不同
# 线性回归的随机参数是靠随机种子通过随机算法生成的，如果不设置随机种子，那每次拿到的数据参数获得的结果每次都不同
from tqdm import tqdm

data, target = load_iris(return_X_y = True)
x = torch.Tensor(data[0: 100])
y = torch.Tensor(target[0: 100]).view(100, 1)

w = torch.randn(1, 4) # 神经元个数1
b = torch.randn(1) # 偏置值

w.requires_grad = True
b.requires_grad = True # 指定回归函数参数可被求导 -> 可训练

def linear(x: torch.Tensor, w:torch.Tensor, b:torch.Tensor) -> torch.Tensor:
    return x @ w.T + b

def sigmoid(y: torch.Tensor) -> torch.Tensor:
    return 1 / (1 + torch.exp(-y))

def binary_cross_entropy(y_pred: torch.Tensor, y_true: torch.Tensor) -> torch.Tensor:
    _loss = -(y_true * torch.log(y_pred) + (1 - y_true) * torch.log(1 - y_pred))
    return _loss.mean()

# 循环执行，不断更新w，b，直到将样本分开。
epoch = 10000 # 训练次数
learning_rate = 0.001 # 学习率， 通常在10的-3到-5次方

tm = tqdm(range(epoch), leave=True, desc="训练中...", colour="blue")
for e in tm:
    # 1. 线性运算，预测
    y_ = linear(x, w, b)
    y_ = sigmoid(y_)
    # 2. 计算误差
    loss = binary_cross_entropy(y_, y)
    # 3. 自动求导
    loss.backward()
    # 4. 梯度下降公式反向传导调整参数
    with torch.no_grad(): # ??
        w -= learning_rate * w.grad
        b -= learning_rate * b.grad
        # ??
        w.grad.zero_()
        b.grad.zero_()
    # 5. 测试更新后的神经元，测准确率
    if e % 1000 == 0:
        # print(f"loss损失值：{loss:10.6f}", end="", flush=True)
        y_[ y_ > 0.5 ] = 1
        y_[ y_ <= 0.5 ] = 0
        current_rate = (y_ == y).float().mean()
        tm.set_postfix({
            "误差损失值": f"{loss:10.5f}",
            "准确率": f"{current_rate * 100:.2f}"
        })
        # print(f"准确率:{current_rate * 100:.2f}")
print("训练完毕")

## torch封装逻辑回归的细节

- torch.nn 神经网络基础封装(面向对象，类)
- torch.nn.functional 神经网络原始封装(面向过程，函数)

### 1. 数据集

In [2]:
from sklearn.datasets import load_iris
import torch
from torch.nn.functional import linear, softmax, sigmoid, binary_cross_entropy

In [3]:
data, target = load_iris(return_X_y = True)
x = torch.tensor(data[0:100], dtype=torch.float32)
y = torch.tensor(target[0:100], dtype=torch.float32).view(-1, 1) # -1表示自动推导形状

In [4]:
x.shape, y.shape

(torch.Size([100, 4]), torch.Size([100, 1]))

### 2. 定义可训练参数

In [5]:
w = torch.randn(1,4, dtype=torch.float32, requires_grad=True)
b = torch.randn(1, dtype=torch.float32, requires_grad=True)

### 3. torch的线性回归矩阵运算

In [6]:
# 测试
y_ = linear(x, w, b)

### 4. torch的sigmoid激活函数

In [7]:
y_ = sigmoid(y_)

### 5. torch的二分交叉熵函数

In [8]:
loss = binary_cross_entropy(y_, y, reduction="mean")

## 6. 训练与梯度下沉

In [9]:
epoch = 100000
lr = 0.001

from tqdm.notebook import tqdm
process_bar = tqdm(range(epoch), desc="执行中...", colour="blue")
for pb in process_bar:
    y_ = linear(x, w, b)
    y_ = sigmoid(y_)
    loss = binary_cross_entropy(y_, y, reduction="mean")
    
    loss.backward()
    with torch.no_grad():
        w -= lr * w.grad
        b -= lr * b.grad
        w.grad.zero_()
        b.grad.zero_()
    if pb % 1000 == 0:
        y_[ y_ > 0.5 ] = 1
        y_[ y_ <= 0.5 ] = 0
        current_rate = (y == y_).float().mean()
        process_bar.set_postfix({
            "损失值:": f"{loss.item()}",
            "正确率:": f"{current_rate * 100:.2f}"
        })
print("训练结束")
    

执行中...:   0%|          | 0/100000 [00:00<?, ?it/s]

训练结束


In [ ]:
from torch.nn import